# Downstream — Hypotension Risk + BP Regression (patch_len = 30)

Standalone notebook that consumes the encoder checkpoint produced by the upstream pretraining notebook (`hybrid_v4_encoders.pth`, trained with `patch_len=30`).

**Pipeline:** compile physiological windows → memmap → 5-fold CV with a multi-task head (binary classification + 15-step MAP/SBP/DBP regression) → temperature-calibrated ensemble evaluation on the held-out test split, swept across forecasting horizons.

**Why a self-contained file?** The model definition relies on `PrivateEncoderRaw`, `PrivateEncoderECG`, `SharedCrossModalEncoder`, and `HybridEncoderBundle` from the upstream architecture — these are inlined below so the notebook runs without importing the pretraining file.

## Cell 0 — Imports and global seed

Everything the downstream pipeline needs in one place: scientific stack, PyTorch + DataLoader utilities, scikit-learn metrics, tqdm, matplotlib. `seed_everything` locks Python / NumPy / PyTorch (CPU + CUDA) RNGs and forces deterministic cuDNN so re-runs are reproducible.

Also defines module-level globals reused by later cells: `SEED`, `DEVICE`, `CANONICAL_CHANNELS = ['ppg','abp','ecg']`, and `TARGET_RECALL`.

In [ ]:
# ==============================================================================
# CELL 0: GLOBAL SEED + IMPORTS
# (from upstream pretraining notebook — ensures determinism across runs)
# ==============================================================================
import os
import random
import time
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import (Dataset, DataLoader, Subset, ConcatDataset,
                              WeightedRandomSampler, random_split)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             f1_score, brier_score_loss,
                             roc_curve, precision_recall_curve)
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')


def seed_everything(seed=42):
    print(f"[*] Locking random seeds to {seed}...")
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("[OK] Seeds locked.")


seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Device: {DEVICE}")

# Module-level globals used throughout the downstream pipeline
SEED = 42
CANONICAL_CHANNELS = ['ppg', 'abp', 'ecg']
TARGET_RECALL = 0.85


## Cell 1 — Hybrid architecture (from upstream)

Inlined verbatim from the pretraining notebook. Three private encoders:
- `PrivateEncoderRaw` for PPG and ABP (linear patch projection → transformer)
- `PrivateEncoderECG` for ECG (learnable stationary wavelet front-end → per-band patches → softmax-weighted band mixture → transformer)

And one shared cross-modal transformer (`SharedCrossModalEncoder`) that attends across all three modalities jointly. `HybridEncoderBundle` is a thin wrapper that runs the three private encoders and the shared one in sequence — its `state_dict` is what `hybrid_v4_encoders.pth` contains.

**Patching:** `patch_len=30` ⇒ `num_patches = 3750 // 30 = 125`. These values are baked into the position embeddings and the `Linear(patch_len, d_model)` projections, so they must match the checkpoint exactly.

In [ ]:
# ==============================================================================
# CELL 1: HYBRID ARCHITECTURE  (imported verbatim from upstream pretraining)
#
# Provides:
#   - LearnableWavelet1D / qmf / _conv_same / DB6_DEC_LO
#   - PrivateEncoderRaw   (PPG, ABP)
#   - PrivateEncoderECG   (wavelet front-end + per-band LayerNorm)
#   - SharedCrossModalEncoder
#   - HybridEncoderBundle
#
# PATCHING:  patch_len = 30  =>  num_patches = 3750 // 30 = 125
# These MUST match what the encoder checkpoint was trained with.
# ==============================================================================

# ── db6 wavelet decomposition coefficients ───────────────────────────────────
DB6_DEC_LO = torch.tensor([
    -0.00107730108499558,
     0.00477725751101065,
     0.000553842200993802,
    -0.031582039318031156,
     0.027522865530016288,
     0.09750160558707936,
    -0.12976686756709563,
    -0.22626469396516913,
     0.31525035170924843,
     0.7511339080215775,
     0.4946238903983854,
     0.11154074335008017,
], dtype=torch.float32)


def qmf(lo: torch.Tensor) -> torch.Tensor:
    """Quadrature-mirror high-pass from low-pass: hi[i] = (-1)^i * lo[N-1-i]."""
    n = lo.shape[-1]
    device = lo.device
    sign = torch.tensor([(-1.0) ** i for i in range(n)],
                        dtype=lo.dtype, device=device)
    return sign * lo.flip(-1)


def _conv_same(x: torch.Tensor, w: torch.Tensor, dilation: int = 1) -> torch.Tensor:
    """1D conv with same-length output via asymmetric padding."""
    f = w.shape[-1]
    pad_total = (f - 1) * dilation
    pad_left  = pad_total // 2
    pad_right = pad_total - pad_left
    x_pad = F.pad(x, (pad_left, pad_right))
    return F.conv1d(x_pad, w, dilation=dilation)


class LearnableWavelet1D(nn.Module):
    """
    Stationary (undecimated) wavelet decomposition with LEARNABLE filters.
    Initialised from db6.  Each level uses dilation 2^level (no decimation).

    Input  : x [B, L]
    Output : bands [B, K, L]   where K = n_levels + 1
             ordering: [D1, D2, ..., D_n_levels, A_n_levels]
    """
    def __init__(self, n_levels: int = 3, init_lo: torch.Tensor = DB6_DEC_LO):
        super().__init__()
        self.n_levels   = n_levels
        self.filter_len = init_lo.shape[-1]
        init_hi         = qmf(init_lo)

        self.lo = nn.ParameterList([
            nn.Parameter(init_lo.clone().view(1, 1, -1)) for _ in range(n_levels)
        ])
        self.hi = nn.ParameterList([
            nn.Parameter(init_hi.clone().view(1, 1, -1)) for _ in range(n_levels)
        ])

    @property
    def num_bands(self) -> int:
        return self.n_levels + 1

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)                                                  # [B,1,L]
        details = []
        current = x
        for lvl in range(self.n_levels):
            d = 2 ** lvl
            lo_out = _conv_same(current, self.lo[lvl], dilation=d)
            hi_out = _conv_same(current, self.hi[lvl], dilation=d)
            details.append(hi_out)
            current = lo_out
        bands = details + [current]
        return torch.cat(bands, dim=1)                                      # [B,K,L]

    def qmf_penalty(self) -> torch.Tensor:
        """MSE between learnt hi and qmf(lo) — anchors filters to a valid basis."""
        loss = 0.0
        for k in range(self.n_levels):
            lo_k = self.lo[k].squeeze()
            hi_k = self.hi[k].squeeze()
            expected_hi = qmf(lo_k)
            loss = loss + F.mse_loss(hi_k, expected_hi)
        return loss / self.n_levels


# ── Private encoder for "raw" modalities (PPG, ABP) ──────────────────────────
class PrivateEncoderRaw(nn.Module):
    def __init__(self, seq_len=3750, patch_len=30, d_model=128,
                 n_heads=8, n_layers=2, dropout=0.1):
        super().__init__()
        self.patch_len   = patch_len
        self.d_model     = d_model
        self.num_patches = seq_len // patch_len

        self.patch_proj = nn.Linear(patch_len, d_model)
        self.pos_embed  = nn.Parameter(
            torch.randn(1, self.num_patches, d_model) * 0.02)
        self.dropout    = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(d_model)

    def forward(self, x):
        patches = x.unfold(dimension=-1, size=self.patch_len, step=self.patch_len)
        z = self.patch_proj(patches) + self.pos_embed
        z = self.dropout(z)
        z = self.transformer(z)
        z = self.norm(z)
        return z


# ── Private encoder for ECG (wavelet front-end + per-band LayerNorm) ─────────
class PrivateEncoderECG(nn.Module):
    """
      raw ECG -> learnable wavelet (K bands)
              -> patch each band -> per-band projection -> per-band LayerNorm
              -> softmax-weighted sum over bands
              -> 2-layer transformer -> z [B, P, d_model]
    """
    def __init__(self, seq_len=3750, patch_len=30, d_model=128,
                 n_heads=8, n_layers=2, dropout=0.1,
                 n_wavelet_levels=3):
        super().__init__()
        self.patch_len    = patch_len
        self.d_model      = d_model
        self.num_patches  = seq_len // patch_len

        self.wavelet = LearnableWavelet1D(n_levels=n_wavelet_levels)
        self.K       = self.wavelet.num_bands

        self.band_proj = nn.ModuleList([
            nn.Linear(patch_len, d_model) for _ in range(self.K)
        ])
        # Per-band LayerNorm — stops one band dominating by scale
        self.band_norm = nn.ModuleList([
            nn.LayerNorm(d_model) for _ in range(self.K)
        ])
        self.band_logits = nn.Parameter(torch.zeros(self.K))

        self.pos_embed = nn.Parameter(
            torch.randn(1, self.num_patches, d_model) * 0.02)
        self.dropout   = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(d_model)

    def forward(self, x):
        bands   = self.wavelet(x)                                          # [B,K,L]
        patches = bands.unfold(dimension=-1, size=self.patch_len,
                               step=self.patch_len)                        # [B,K,P,patch_len]

        proj = []
        for k in range(self.K):
            p_k = self.band_proj[k](patches[:, k])                         # [B,P,D]
            p_k = self.band_norm[k](p_k)
            proj.append(p_k)
        proj = torch.stack(proj, dim=1)                                    # [B,K,P,D]

        w = F.softmax(self.band_logits, dim=0)                             # [K]
        z = (proj * w.view(1, -1, 1, 1)).sum(dim=1)                        # [B,P,D]

        z = z + self.pos_embed
        z = self.dropout(z)
        z = self.transformer(z)
        z = self.norm(z)
        return z


# ── Shared cross-modal encoder ───────────────────────────────────────────────
class SharedCrossModalEncoder(nn.Module):
    def __init__(self, d_model=128, n_heads=8, n_layers=3,
                 dropout=0.1, num_patches=125):
        super().__init__()
        self.d_model        = d_model
        self.num_patches    = num_patches
        self.modality_embed = nn.Embedding(3, d_model)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm        = nn.LayerNorm(d_model)

    def forward(self, z_ppg, z_abp, z_ecg):
        B, P, D = z_ppg.shape
        device  = z_ppg.device

        m_ppg = self.modality_embed(torch.zeros(1, dtype=torch.long, device=device)).view(1,1,D)
        m_abp = self.modality_embed(torch.ones(1,  dtype=torch.long, device=device)).view(1,1,D)
        m_ecg = self.modality_embed(torch.full((1,), 2, dtype=torch.long, device=device)).view(1,1,D)

        z = torch.cat([z_ppg + m_ppg, z_abp + m_abp, z_ecg + m_ecg], dim=1) # [B, 3P, D]
        z = self.transformer(z)
        z = self.norm(z)
        return z[:, 0:P], z[:, P:2*P], z[:, 2*P:3*P]


class HybridEncoderBundle(nn.Module):
    def __init__(self, enc_ppg, enc_abp, enc_ecg, shared):
        super().__init__()
        self.enc_ppg = enc_ppg
        self.enc_abp = enc_abp
        self.enc_ecg = enc_ecg
        self.shared  = shared

    def forward(self, x_ppg, x_abp, x_ecg):
        z_p = self.enc_ppg(x_ppg)
        z_a = self.enc_abp(x_abp)
        z_e = self.enc_ecg(x_ecg)
        return self.shared(z_p, z_a, z_e)


print("[OK] Architecture loaded  (patch_len=30, num_patches=125).")


## Cell 2 — Compile downstream physio data into memmap

Walks `Master_Training_Data_Expanded_New_2/{split}/{ppg,abp,ecg}/{offset_*m}/` and packs each window's waveforms, binary labels, and scalar features (SpO₂, HR, horizon) into per-(split, window) memmaps. BP targets (15-step SBP/DBP/MAP) come from the separate `BP_target/` folders — one fixed near-event value per case, reused across all input windows. Missing BP files fall back to clinical defaults (120/80/65 mmHg) and are reported.

Skips automatically if `n_samples.npy` already exists in the output dir, so re-running is cheap.

In [ ]:
# ==============================================================================
# CELL 10: COMPILE DOWNSTREAM PHYSIO DATA INTO MEMMAP
# BP targets (SBP / DBP / MAP) now read from the SEPARATE BP_target folders
# (fixed near-event -0m30s target per case), NOT from each window's abp _based.csv.
# ==============================================================================
import os
import numpy as np
import pandas as pd
from tqdm import tqdm


def compile_physio_to_memmap(data_dir, output_dir, bp_target_dir,
                             splits, windows, seq_len=3750):
    """
    Waveforms + scalars come from each window's CSVs; SBP/DBP/MAP targets come
    from the standalone BP_target/{sbp,dbp,map}_target folders (one fixed
    near-event value per case, identical across all input windows).
    """
    os.makedirs(output_dir, exist_ok=True)

    sbp_dir = os.path.join(bp_target_dir, 'sbp_target')
    dbp_dir = os.path.join(bp_target_dir, 'dbp_target')
    map_dir = os.path.join(bp_target_dir, 'map_target')

    def safe_extract(df):
        val = df.iloc[0, 6:].values.astype(np.float32)
        m   = np.nanmean(val) if not np.all(np.isnan(val)) else 0.0
        val = np.nan_to_num(val, nan=m)
        if len(val) >= seq_len:
            return val[:seq_len]
        return np.pad(val, (0, seq_len - len(val)))

    def load_bp_target(path, prefix, default_fill, n=15):
        """Read 15 BP target points from a BP_target CSV; returns array / 100."""
        vals = np.full(n, default_fill, dtype=np.float32)
        if not os.path.exists(path):
            return vals / 100.0
        try:
            df = pd.read_csv(path)
        except Exception:
            return vals / 100.0
        if df.empty:
            return vals / 100.0
        cols = sorted(
            [c for c in df.columns
             if c.startswith(f'{prefix}_') and c.rsplit('_', 1)[-1].isdigit()],
            key=lambda c: int(c.rsplit('_', 1)[-1]))
        if not cols:
            return vals / 100.0
        extracted = df.iloc[0][cols[:n]].values.astype(np.float32)
        vals[:len(extracted)] = extracted
        fill = float(np.nanmean(vals))
        if np.isnan(fill):
            fill = default_fill
        return np.nan_to_num(vals, nan=fill) / 100.0

    for split in splits:
        for window in windows:
            out_dir = os.path.join(output_dir, split, window)
            skip_flag = os.path.join(out_dir, 'n_samples.npy')
            if os.path.exists(skip_flag):
                n = int(np.load(skip_flag)[0])
                print(f"[*] Already compiled: {split}/{window}  ({n} samples). Skipping.")
                continue

            abp_dir = os.path.join(data_dir, split, 'abp', window)
            ppg_dir = os.path.join(data_dir, split, 'ppg', window)
            ecg_dir = os.path.join(data_dir, split, 'ecg', window)
            os.makedirs(out_dir, exist_ok=True)

            abp_files = sorted([f for f in os.listdir(abp_dir) if f.endswith('_wave.csv')])
            N = len(abp_files)
            print(f"[*] Compiling split={split}, window={window}, N={N}")

            horizon_min = float(window.replace('offset_', '').replace('m', ''))

            ppg_mm = np.memmap(os.path.join(out_dir, 'ppg.dat'), dtype='float32', mode='w+', shape=(N, seq_len))
            abp_mm = np.memmap(os.path.join(out_dir, 'abp.dat'), dtype='float32', mode='w+', shape=(N, seq_len))
            ecg_mm = np.memmap(os.path.join(out_dir, 'ecg.dat'), dtype='float32', mode='w+', shape=(N, seq_len))
            lbl_mm = np.memmap(os.path.join(out_dir, 'labels.dat'), dtype='int32',   mode='w+', shape=(N,))
            sc_mm  = np.memmap(os.path.join(out_dir, 'scalars.dat'), dtype='float32', mode='w+', shape=(N, 3))
            map_mm = np.memmap(os.path.join(out_dir, 'map_targets.dat'), dtype='float32', mode='w+', shape=(N, 15))
            sbp_mm = np.memmap(os.path.join(out_dir, 'sbp_targets.dat'), dtype='float32', mode='w+', shape=(N, 15))
            dbp_mm = np.memmap(os.path.join(out_dir, 'dbp_targets.dat'), dtype='float32', mode='w+', shape=(N, 15))

            missing_bp = 0

            for i, fname in enumerate(tqdm(abp_files)):
                df_abp = pd.read_csv(os.path.join(abp_dir, fname))
                df_ppg = pd.read_csv(os.path.join(ppg_dir, fname.replace('abp', 'ppg')))
                df_ecg = pd.read_csv(os.path.join(ecg_dir, fname.replace('abp', 'ecg')))

                # per-window abp based csv -> used ONLY for spo2/hr scalars (see note below)
                based_path = os.path.join(abp_dir, fname.replace('_wave.csv', '_based.csv'))
                df_based = (pd.read_csv(based_path) if os.path.exists(based_path) else pd.DataFrame())

                abp_mm[i] = safe_extract(df_abp)
                ppg_mm[i] = safe_extract(df_ppg)
                ecg_mm[i] = safe_extract(df_ecg)
                lbl_mm[i] = int(df_abp.iloc[0]['label_binary'])

                spo2_val, hr_val = 98.0, 80.0
                if not df_based.empty:
                    spo2_cols = [c for c in df_based.columns if c.startswith('pleth_spo2')]
                    hr_cols   = [c for c in df_based.columns if c.startswith('pleth_hr')]
                    if spo2_cols:
                        spo2_val = float(np.nanmean(df_based.iloc[0][spo2_cols].values.astype(float)))
                    if hr_cols:
                        hr_val = float(np.nanmean(df_based.iloc[0][hr_cols].values.astype(float)))

                sc_mm[i] = [
                    np.nan_to_num(spo2_val, nan=98.0) / 100.0,
                    np.nan_to_num(hr_val,   nan=80.0) / 100.0,
                    horizon_min / 15.0,
                ]

                # ---- BP TARGETS from standalone BP_target folders ----
                # abp wave filename -> case stem (strip split prefix + suffix)
                stem = fname.replace('_abp_wave.csv', '')
                ci = stem.find('Case_')
                case_stem = stem[ci:] if ci >= 0 else stem

                sbp_path = os.path.join(sbp_dir, f"{case_stem}_sbp.csv")
                dbp_path = os.path.join(dbp_dir, f"{case_stem}_dbp.csv")
                map_path = os.path.join(map_dir, f"{case_stem}_map.csv")

                if not os.path.exists(sbp_path):
                    missing_bp += 1

                map_mm[i] = load_bp_target(map_path, 'map', 65.0)
                sbp_mm[i] = load_bp_target(sbp_path, 'sbp', 120.0)
                dbp_mm[i] = load_bp_target(dbp_path, 'dbp', 80.0)

            del ppg_mm, abp_mm, ecg_mm, lbl_mm, sc_mm, map_mm, sbp_mm, dbp_mm
            np.save(os.path.join(out_dir, 'n_samples.npy'), np.array([N]))
            np.save(os.path.join(out_dir, 'filenames.npy'), np.array(abp_files))
            if missing_bp:
                print(f"    [!] {missing_bp}/{N} samples had NO BP target file -> default-filled.")
            print(f"    [OK] {out_dir}\n")


# ── Execute ───────────────────────────────────────────────────────────────────
DOWNSTREAM_DATA_DIR = ("/kaggle/input/datasets/phantom8/thesis-dataset-extended-new-2/Master_Training_Data_Expanded_New_2")
PHYSIO_MEMMAP_DIR   = "/kaggle/working/physio_memmap"
BP_TARGET_DIR       = "/kaggle/input/datasets/phantom8/bp-target-with-2134-control/BP_target"   # <-- VERIFY: must contain sbp_target/ dbp_target/ map_target/
TARGET_WINDOWS      = ['offset_3m', 'offset_5m', 'offset_7m', 'offset_10m', 'offset_12m','offset_15m']

compile_physio_to_memmap(
    data_dir    = DOWNSTREAM_DATA_DIR,
    output_dir  = PHYSIO_MEMMAP_DIR,
    bp_target_dir = BP_TARGET_DIR,
    splits      = ['train', 'val', 'test'],
    windows     = TARGET_WINDOWS,
    seq_len     = 3750,
)
print("[OK] Downstream physio memmap complete.")

## Cell 3 — Waveform augmentation (positives only)

Physiologically-plausible per-channel augmentations applied to robust-z waveforms: jitter, global amplitude scale, smooth magnitude warp, smooth time warp, and low-frequency baseline drift. Each transform fires independently with its own probability so positive samples see varied views.

**Why minority-class only:** with ~100 positives, oversampling without perturbation makes the model memorise the few positive shapes. SMOTE is deliberately avoided — interpolating raw PPG/ABP/ECG between patients is not physiologically meaningful.

In [ ]:
# ==============================================================================
# CELL 11a: WAVEFORM AUGMENTATION  (CHANGE 5 — minority-class only)
# ==============================================================================
import os
import numpy as np
import torch
from torch.utils.data import Dataset, ConcatDataset
from sklearn.model_selection import StratifiedKFold
from collections import Counter
 
CANONICAL_CHANNELS = ['ppg', 'abp', 'ecg']
def _smooth_curve(L, n_knots, sigma, rng, center=1.0):
    """Smooth random multiplicative curve of length L, centred on `center`."""
    knots = rng.normal(center, sigma, size=n_knots)
    xk = np.linspace(0, L - 1, n_knots)
    return np.interp(np.arange(L), xk, knots).astype(np.float32)


def augment_waveform(wave_t, rng,
                     p_jitter=0.6, jitter_std=0.05,
                     p_scale=0.5, scale_std=0.05,
                     p_magwarp=0.5, magwarp_sigma=0.10,
                     p_timewarp=0.4, timewarp_sigma=0.10,
                     p_baseline=0.4, baseline_amp=0.10):
    """
    Physiologically-plausible augmentation on a (C, L) robust-z waveform tensor.
    Each transform fires independently so positives see varied views.
    Applied AFTER robust-z (signals ~unit scale, clipped +/-5), so magnitudes
    are kept small on purpose. SMOTE is intentionally NOT used — interpolating
    raw PPG/ABP/ECG between patients is not physiologically meaningful.
    """
    w = wave_t.numpy().copy()                       # (C, L)
    C, L = w.shape

    for c in range(C):
        sig = w[c]

        if rng.random() < p_jitter:                 # additive noise
            sig = sig + rng.normal(0.0, jitter_std, size=L).astype(np.float32)

        if rng.random() < p_scale:                  # global amplitude scale
            sig = sig * np.float32(rng.normal(1.0, scale_std))

        if rng.random() < p_magwarp:                # smooth amplitude warp
            sig = sig * _smooth_curve(L, 4, magwarp_sigma, rng, center=1.0)

        if rng.random() < p_timewarp:               # smooth time warp
            curve = _smooth_curve(L, 4, timewarp_sigma, rng, center=1.0)
            curve = np.clip(curve, 0.1, None)
            cum = np.cumsum(curve)
            cum = (cum - cum.min()) / (cum.max() - cum.min() + 1e-8) * (L - 1)
            sig = np.interp(np.arange(L), cum, sig).astype(np.float32)

        if rng.random() < p_baseline:               # low-frequency wander
            sig = sig + (_smooth_curve(L, 3, baseline_amp, rng, center=0.0))

        w[c] = sig

    return torch.from_numpy(w.astype(np.float32))

## Cell 4 — Memmap-backed dataset and 5-fold CV pool builder

`PhysiologicalDatasetMM` reads the memmaps produced by Cell 2 and returns `((waveforms_3ch, scalars), (label, map_target, sbp_target, dbp_target))`. Robust-z is applied per segment at item-read time so the stored data stays raw.

`AugmentingSubset` wraps a `Subset` to augment **only positive** samples and **only when `augment=True`** — used for the training fold; the validation fold wraps the same pool with `augment=False` so eval stays clean.

`build_cv_pools_mm` pools `train + val` for 5-fold StratifiedKFold and leaves `test` untouched for the final ensemble evaluation.

In [ ]:
# ==============================================================================
# CELL 11b: DATASET + AUGMENTING WRAPPER + CV POOL BUILDER
# ==============================================================================
class PhysiologicalDatasetMM(Dataset):
    """Memmap-backed dataset — returns 3 channels [ppg, abp, ecg], robust-z."""
    def __init__(self, memmap_dir, split, window):
        super().__init__()
        d = os.path.join(memmap_dir, split, window)
        N = int(np.load(os.path.join(d, 'n_samples.npy'))[0])
        self.N = N

        self.waves = {}
        for ch in CANONICAL_CHANNELS:
            self.waves[ch] = np.memmap(
                os.path.join(d, f'{ch}.dat'),
                dtype='float32', mode='r', shape=(N, 3750))

        self.labels = np.memmap(os.path.join(d, 'labels.dat'),
                                dtype='int32', mode='r', shape=(N,))
        self.scalars = np.memmap(os.path.join(d, 'scalars.dat'),
                                 dtype='float32', mode='r', shape=(N, 3))
        self.map_tgts = np.memmap(os.path.join(d, 'map_targets.dat'),
                                  dtype='float32', mode='r', shape=(N, 15))
        self.sbp_tgts = np.memmap(os.path.join(d, 'sbp_targets.dat'),
                                  dtype='float32', mode='r', shape=(N, 15))
        self.dbp_tgts = np.memmap(os.path.join(d, 'dbp_targets.dat'),
                                  dtype='float32', mode='r', shape=(N, 15))

        pos_rate = float(np.mean(self.labels))
        print(f"  [{split.upper()} | {window}]  {N} samples  pos_rate={pos_rate:.3f}")

    def __len__(self):
        return self.N

    @staticmethod
    def _robust_z(x, clip=5.0):
        med = np.median(x)
        q75, q25 = np.percentile(x, [75, 25])
        iqr = (q75 - q25) + 1e-6
        z = np.clip((x - med) / iqr, -clip, clip)
        return z.astype(np.float32)

    def __getitem__(self, idx):
        wave_list = [torch.tensor(self._robust_z(self.waves[ch][idx].copy()),
                                  dtype=torch.float32)
                     for ch in CANONICAL_CHANNELS]
        waveforms_t = torch.stack(wave_list, dim=0)             # (3, 3750)

        scalars_t = torch.tensor(self.scalars[idx].copy(), dtype=torch.float32)
        map_t = torch.tensor(self.map_tgts[idx].copy(), dtype=torch.float32)
        sbp_t = torch.tensor(self.sbp_tgts[idx].copy(), dtype=torch.float32)
        dbp_t = torch.tensor(self.dbp_tgts[idx].copy(), dtype=torch.float32)
        label_t = torch.tensor(float(self.labels[idx]), dtype=torch.float32)

        return (waveforms_t, scalars_t), (label_t, map_t, sbp_t, dbp_t)


class AugmentingSubset(Dataset):
    """
    Subset that augments POSITIVE samples only, and only when augment=True.
    Use augment=True for the TRAIN subset, augment=False for VAL — both wrap
    the same shared pooled dataset without contaminating validation.
    """
    def __init__(self, base, indices, augment=False, aug_prob=0.5, seed=SEED):
        self.base = base
        self.indices = list(indices)
        self.augment = augment
        self.aug_prob = aug_prob
        self.rng = np.random.RandomState(seed)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        (wave, scalars), (label, map_t, sbp_t, dbp_t) = self.base[idx]
        if (self.augment and label.item() > 0.5
                and self.rng.random() < self.aug_prob):
            wave = augment_waveform(wave, self.rng)
        return (wave, scalars), (label, map_t, sbp_t, dbp_t)


def build_cv_pools_mm(memmap_dir, window, n_folds=5):
    train_ds = PhysiologicalDatasetMM(memmap_dir, 'train', window)
    val_ds = PhysiologicalDatasetMM(memmap_dir, 'val', window)
    test_ds = PhysiologicalDatasetMM(memmap_dir, 'test', window)

    pooled_ds = ConcatDataset([train_ds, val_ds])
    pooled_labels = ([int(x) for x in train_ds.labels] +
                     [int(x) for x in val_ds.labels])

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    folds = list(skf.split(np.zeros(len(pooled_labels)), pooled_labels))

    counts = Counter(pooled_labels)
    print(f"  Pooled CV: {len(pooled_ds)} | Test: {len(test_ds)} "
          f"| neg={counts[0]} pos={counts[1]}")
    return pooled_ds, pooled_labels, folds, test_ds

## Cell 5 — Configuration

All downstream hyper-parameters. The architecture section must match the pretraining run — `patch_len=30`, `d_model=128`, `n_heads=8`, `private_layers=2`, `shared_layers=3`, `n_wavelet_levels=3` — otherwise the encoder checkpoint will fail to load.

Loss balance: classification (focal, γ=2, α=0.5) plus regression (Huber, δ=0.05 in normalised units ≈ 5 mmHg) weighted by `alpha=0.5, beta=2.0`. Imbalance is handled by a `WeightedRandomSampler` targeting 30 % positives per batch, plus minority-class augmentation. Probabilities are temperature-calibrated on pooled OOF logits before the final ensemble evaluation.

In [ ]:
# ==============================================================================
# CELL 12: CONFIG
# ==============================================================================
PHYSIO_MEMMAP_DIR = "/kaggle/input/datasets/phantom8/downtream-hybrid-ckpts-ppg-abp-ecg/physio_memmap"
ENCODER_PATH = ('/kaggle/input/datasets/phantom8/pretrain-v6-weights/'
                'hybrid_v4_encoders.pth')

TARGET_WINDOWS = ['offset_3m', 'offset_5m', 'offset_7m','offset_12m',
                  'offset_10m', 'offset_15m']          

CONFIG = dict(
    physio_memmap_dir=PHYSIO_MEMMAP_DIR,
    foundation_path=ENCODER_PATH,
    n_folds=5,
    batch_size=16,
    select_w_auprc=0.5,    # selection metric: AUPRC weight (was AUROC)
    select_w_reg=0.5,      # selection metric: regression-r weight
    epochs=60,
    patience=12,
    lr_encoder=1e-5,
    lr_heads=1e-4,
    weight_decay=5e-4,

    # ── Multi-task loss balance ──────────────────────────────────────────────
    alpha=0.5,             # classification weight in the total loss
    beta=2.0,              # regression weight
    huber_delta=0.05,      # ~5 mmHg linear region

    # ── Imbalance handling (CHANGES 3 & 4) ───────────────────────────────────
    cls_loss='focal',      # 'focal' or 'weighted_bce'
    focal_gamma=2.0,       # hard-example focus; cap at 2 with ~100 positives
    focal_alpha=0.50,      # NEUTRAL — the sampler does the class balancing
    pos_weight_cap=None,   # only used on the 'weighted_bce' path

    use_sampler=True,      # WeightedRandomSampler on the train loader
    sampler_pos_frac=0.30, # target positive fraction per batch (partial)

    # ── Augmentation (CHANGE 5) ──────────────────────────────────────────────
    augment_positives=True,
    aug_prob=0.5,          # P(apply aug) per positive TRAIN sample

    # ── Calibration (CHANGE 1) ───────────────────────────────────────────────
    calibrate=True,        # temperature scaling on pooled OOF logits

    # ── Hybrid architecture knobs (MUST match pretraining) ──────────────────
    n_bp_steps=15,
    seq_len=3750,
    patch_len=30,
    d_model=128,
    n_heads=8,
    private_layers=2,
    shared_layers=3,
    n_wavelet_levels=3,
    num_scalars=3,
    dropout=0.1,
)

## Cell 6 — Multi-task model with pretrained encoder

`PretrainedMultiTaskHybrid` loads `hybrid_v4_encoders.pth` into a fresh `HybridEncoderBundle` (`strict=False` so any small key drift is reported, not fatal), then attaches four heads on top of the patch-pooled fused features:

- a single-logit hypotension classifier
- three independent 15-step regression heads for MAP, SBP, DBP

Head biases are initialised to physiological priors (MAP≈0.70, SBP≈1.20, DBP≈0.70 in normalised units) so training starts near a sensible BP range.

In [ ]:
# ==============================================================================
# CELL 13: HYBRID MULTI-TASK MODEL  (logic unchanged from original)
#
# Requires PrivateEncoderRaw / PrivateEncoderECG / SharedCrossModalEncoder /
# HybridEncoderBundle to be in scope (import from the pretraining module).
# ==============================================================================
class PretrainedMultiTaskHybrid(nn.Module):
    CHANNEL_INDEX = {'ppg': 0, 'abp': 1, 'ecg': 2}

    def __init__(self, foundation_path,
                 d_model=128, patch_len=30, seq_len=3750, n_heads=8,
                 private_layers=2, shared_layers=3, n_wavelet_levels=3,
                 num_scalars=3, n_bp_steps=15, dropout=0.1):
        super().__init__()
        self.n_channels = 3
        self.n_bp_steps = n_bp_steps
        self.d_model = d_model

        self.enc_ppg = PrivateEncoderRaw(
            seq_len=seq_len, patch_len=patch_len, d_model=d_model,
            n_heads=n_heads, n_layers=private_layers, dropout=dropout)
        self.enc_abp = PrivateEncoderRaw(
            seq_len=seq_len, patch_len=patch_len, d_model=d_model,
            n_heads=n_heads, n_layers=private_layers, dropout=dropout)
        self.enc_ecg = PrivateEncoderECG(
            seq_len=seq_len, patch_len=patch_len, d_model=d_model,
            n_heads=n_heads, n_layers=private_layers, dropout=dropout,
            n_wavelet_levels=n_wavelet_levels)
        self.shared = SharedCrossModalEncoder(
            d_model=d_model, n_heads=n_heads, n_layers=shared_layers,
            dropout=dropout, num_patches=seq_len // patch_len)

        self.encoder = HybridEncoderBundle(
            self.enc_ppg, self.enc_abp, self.enc_ecg, self.shared)

        if os.path.exists(foundation_path):
            state = torch.load(foundation_path, map_location='cpu')
            missing, unexpected = self.encoder.load_state_dict(state, strict=False)
            print(f"  [OK] Loaded foundation: {foundation_path}")
            if missing:
                print(f"  [warn] Missing keys ({len(missing)}): {missing[:3]}"
                      f"{'...' if len(missing) > 3 else ''}")
            if unexpected:
                print(f"  [warn] Unexpected keys ({len(unexpected)}): "
                      f"{unexpected[:3]}{'...' if len(unexpected) > 3 else ''}")
        else:
            print(f"  [!] Foundation not found: {foundation_path}"
                  " — training encoder from scratch.")

        fused_dim = self.n_channels * d_model + num_scalars
        self.proj = nn.Sequential(
            nn.Linear(fused_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Dropout(dropout),
        )

        self.classifier = nn.Linear(256, 1)

        def _reg_head():
            return nn.Sequential(
                nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout / 2),
                nn.Linear(128, n_bp_steps))

        self.map_regressor = _reg_head()
        self.sbp_regressor = _reg_head()
        self.dbp_regressor = _reg_head()

        self._init_heads()
        with torch.no_grad():
            self.map_regressor[-1].bias.fill_(0.70)
            self.sbp_regressor[-1].bias.fill_(1.20)
            self.dbp_regressor[-1].bias.fill_(0.70)

    def _init_heads(self):
        for name, m in self.named_modules():
            if name.startswith('encoder'):
                continue
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, waveforms, scalars=None):
        x_ppg = waveforms[:, self.CHANNEL_INDEX['ppg'], :]
        x_abp = waveforms[:, self.CHANNEL_INDEX['abp'], :]
        x_ecg = waveforms[:, self.CHANNEL_INDEX['ecg'], :]

        z_ppg_s, z_abp_s, z_ecg_s = self.encoder(x_ppg, x_abp, x_ecg)
        h_ppg = z_ppg_s.mean(dim=1)
        h_abp = z_abp_s.mean(dim=1)
        h_ecg = z_ecg_s.mean(dim=1)
        patient_features = torch.cat([h_ppg, h_abp, h_ecg], dim=1)

        fused = (torch.cat([patient_features, scalars], dim=1)
                 if scalars is not None else patient_features)
        shared = self.proj(fused)

        cls_logit = self.classifier(shared).squeeze(-1)
        return (cls_logit, self.map_regressor(shared),
                self.sbp_regressor(shared), self.dbp_regressor(shared))


def build_model(cfg):
    return PretrainedMultiTaskHybrid(
        foundation_path=cfg['foundation_path'], d_model=cfg['d_model'],
        patch_len=cfg['patch_len'], seq_len=cfg['seq_len'], n_heads=cfg['n_heads'],
        private_layers=cfg['private_layers'], shared_layers=cfg['shared_layers'],
        n_wavelet_levels=cfg['n_wavelet_levels'], num_scalars=cfg['num_scalars'],
        n_bp_steps=cfg['n_bp_steps'], dropout=cfg['dropout']).to(DEVICE)

## Cell 7 — Losses, metrics, calibration

`FocalLossBCE` (binary focal loss with α/γ), `MultiTaskLoss` (focal + 3× Huber, weighted), and `build_cls_criterion` (single place that prevents double-correction across focal-α, pos-weight, and the sampler).

Threshold pick: **Youden's J** on the ROC of pooled OOF probabilities. Calibration: **temperature scaling** (LBFGS over a single `log_T` parameter) on pooled OOF logits — chosen over isotonic because the positive count is too small for isotonic to be reliable.

Evaluation: per-fold validation pass returns raw logits; the final ensemble passes average **calibrated** probabilities across folds and also averages the BP regressions. Bootstrap CIs (1 000 resamples) are reported for AUROC, AUPRC, F1, Brier.

In [ ]:
# ==============================================================================
# CELL 14: LOSSES, METRICS, CALIBRATION, EVAL
# ==============================================================================
class FocalLossBCE(nn.Module):
    """Binary focal loss (CHANGE 3).  loss = alpha_t * (1 - p_t)^gamma * BCE."""
    def __init__(self, alpha=0.5, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = torch.exp(-bce)                                   # prob of true class
        a_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


class MultiTaskLoss(nn.Module):
    """L_total = alpha * L_cls + beta * (L_map + L_sbp + L_dbp)  [Huber]."""
    def __init__(self, cls_criterion, alpha=0.5, beta=2.0, huber_delta=0.05):
        super().__init__()
        self.cls_loss = cls_criterion
        self.huber = nn.HuberLoss(delta=huber_delta)
        self.alpha = alpha
        self.beta = beta

    def forward(self, cls_logit, map_p, sbp_p, dbp_p,
                cls_t, map_t, sbp_t, dbp_t):
        l_cls = self.cls_loss(cls_logit, cls_t)
        l_map = self.huber(map_p, map_t)
        l_sbp = self.huber(sbp_p, sbp_t)
        l_dbp = self.huber(dbp_p, dbp_t)
        total = self.alpha * l_cls + self.beta * (l_map + l_sbp + l_dbp)
        return total, l_cls, l_map, l_sbp, l_dbp


def build_cls_criterion(cfg, fold_train_labels):
    """
    Single place that prevents DOUBLE-CORRECTION across the three positive-
    boosting knobs (focal alpha / pos_weight / sampler):

      - focal + sampler      -> alpha kept neutral (sampler balances batches)
      - focal alone          -> alpha bumped up to emphasise positives
      - weighted_bce + sampler -> pos_weight square-rooted (tempered)
      - weighted_bce alone   -> raw neg/pos ratio (optionally capped)
    """
    if cfg['cls_loss'] == 'focal':
        alpha = cfg['focal_alpha']
        if not cfg['use_sampler']:
            alpha = max(alpha, 0.75)          # no sampler -> lean on alpha
        print(f"    [loss] focal  gamma={cfg['focal_gamma']}  alpha={alpha:.2f}"
              f"  (sampler={'on' if cfg['use_sampler'] else 'off'})")
        return FocalLossBCE(alpha=alpha, gamma=cfg['focal_gamma'])

    counts = Counter(fold_train_labels)
    ratio = counts[0] / max(counts[1], 1)
    if cfg['use_sampler']:
        ratio = ratio ** 0.5                   # temper: batches already balanced
    if cfg['pos_weight_cap'] is not None:
        ratio = min(ratio, cfg['pos_weight_cap'])
    pw = torch.tensor([ratio], dtype=torch.float32, device=DEVICE)
    print(f"    [loss] weighted_bce  pos_weight={ratio:.3f}  "
          f"(neg={counts[0]}, pos={counts[1]}, "
          f"sampler={'on' if cfg['use_sampler'] else 'off'})")
    return nn.BCEWithLogitsLoss(pos_weight=pw)


def make_train_loader(pooled_ds, train_idx, fold_train_labels, cfg):
    train_subset = AugmentingSubset(
        pooled_ds, train_idx,
        augment=cfg['augment_positives'], aug_prob=cfg['aug_prob'])

    if cfg['use_sampler']:                     # CHANGE 4: partial oversampling
        labels_arr = np.asarray(fold_train_labels)
        n_pos = max(int((labels_arr == 1).sum()), 1)
        n_neg = max(int((labels_arr == 0).sum()), 1)
        f = cfg['sampler_pos_frac']
        w_pos = f * n_neg / ((1 - f) * n_pos)  # solve E[pos fraction] = f
        sample_w = np.where(labels_arr == 1, w_pos, 1.0).astype(np.float64)
        sampler = WeightedRandomSampler(
            torch.as_tensor(sample_w), num_samples=len(sample_w),
            replacement=True)
        print(f"    [sampler] target pos_frac={f:.2f}  w_pos={w_pos:.2f}")
        return DataLoader(train_subset, batch_size=cfg['batch_size'],
                          sampler=sampler, drop_last=True,
                          num_workers=2, pin_memory=True)

    return DataLoader(train_subset, batch_size=cfg['batch_size'],
                      shuffle=True, drop_last=True,
                      num_workers=2, pin_memory=True)


def pearson_r(pred, target):
    p = pred.flatten().float()
    t = target.flatten().float()
    vp, vt = p - p.mean(), t - t.mean()
    den = torch.sqrt((vp ** 2).sum() * (vt ** 2).sum()) + 1e-8
    return ((vp * vt).sum() / den).item()


def pick_threshold_youden(labels, probs):
    """Youden's J: pick threshold that maximizes (TPR - FPR) on the ROC curve."""
    labels = np.asarray(labels).astype(int)
    probs = np.asarray(probs).astype(float)
    if len(np.unique(labels)) < 2:
        return 0.5, dict(method='degenerate')

    fpr, tpr, roc_thresh = roc_curve(labels, probs)
    j = tpr - fpr
    best = int(np.argmax(j))
    chosen = float(roc_thresh[best])
    if not np.isfinite(chosen):                       # roc_curve sometimes emits +inf at idx 0
        chosen = float(probs.max() + 1e-6)
    return chosen, dict(method='youden_j',
                        j=float(j[best]),
                        tpr_at_thr=float(tpr[best]),
                        fpr_at_thr=float(fpr[best]))

def compute_cls_metrics(labels, probs, threshold):
    preds = (probs >= threshold).astype(int)
    two = len(np.unique(labels)) > 1
    return dict(
        auroc=roc_auc_score(labels, probs) if two else 0.5,
        auprc=average_precision_score(labels, probs) if two else 0.5,
        f1=f1_score(labels, preds, zero_division=0),
        brier=brier_score_loss(labels, probs),
        threshold=threshold)


def bootstrap_ci(labels, probs, threshold, n_bootstrap=1000, ci=0.95, seed=SEED):
    rng = np.random.RandomState(seed)
    n = len(labels)
    names = ['auroc', 'auprc', 'f1', 'brier']
    boot = {m: [] for m in names}
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        bl, bp = labels[idx], probs[idx]
        if len(np.unique(bl)) < 2:
            continue
        boot['auroc'].append(roc_auc_score(bl, bp))
        boot['auprc'].append(average_precision_score(bl, bp))
        boot['f1'].append(f1_score(bl, (bp >= threshold).astype(int),
                                   zero_division=0))
        boot['brier'].append(brier_score_loss(bl, bp))
    alpha = (1 - ci) / 2
    out = {}
    for k in names:
        arr = np.array(boot[k])
        if len(arr) == 0:
            out[f'{k}_lo'] = out[f'{k}_hi'] = float('nan')
        else:
            out[f'{k}_lo'] = float(np.percentile(arr, 100 * alpha))
            out[f'{k}_hi'] = float(np.percentile(arr, 100 * (1 - alpha)))
    return out


def fit_temperature(logits, labels, max_iter=200):
    """
    CHANGE 1: temperature scaling. One parameter, fit by NLL on POOLED OOF
    logits — robust at low positive counts (isotonic would overfit here).
    """
    logits = torch.as_tensor(np.asarray(logits), dtype=torch.float32)
    labels = torch.as_tensor(np.asarray(labels), dtype=torch.float32)
    log_T = torch.zeros(1, requires_grad=True)        # T = exp(log_T) > 0
    opt = torch.optim.LBFGS([log_T], lr=0.05, max_iter=max_iter)
    bce = nn.BCEWithLogitsLoss()

    def closure():
        opt.zero_grad()
        loss = bce(logits / torch.exp(log_T), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.exp(log_T).detach().clamp(0.05, 20.0).item())


@torch.no_grad()
def _val_forward(model, loader, device):
    """Return raw logits (not probs) plus BP tensors for one pass."""
    model.eval()
    logits_l, labels_l = [], []
    mp_l, mt_l, sp_l, st_l, dp_l, dt_l = [], [], [], [], [], []
    for (wave, scal), (labels, map_t, sbp_t, dbp_t) in loader:
        wave, scal = wave.to(device), scal.to(device)
        logit, mp, sp, dp = model(wave, scal)
        logits_l.extend(logit.cpu().numpy())
        labels_l.extend(labels.cpu().numpy())
        mp_l.append(mp.cpu()); mt_l.append(map_t)
        sp_l.append(sp.cpu()); st_l.append(sbp_t)
        dp_l.append(dp.cpu()); dt_l.append(dbp_t)
    return (np.array(labels_l), np.array(logits_l),
            torch.cat(mp_l), torch.cat(mt_l),
            torch.cat(sp_l), torch.cat(st_l),
            torch.cat(dp_l), torch.cat(dt_l))


@torch.no_grad()
def evaluate_ensemble(models, loader, device, temperature, threshold):
    """CHANGE 2: average calibrated probabilities + BP regressions over folds."""
    for m in models:
        m.eval()
    k = len(models)
    probs_l, labels_l = [], []
    mp_l, mt_l, sp_l, st_l, dp_l, dt_l = [], [], [], [], [], []

    for (wave, scal), (labels, map_t, sbp_t, dbp_t) in loader:
        wave, scal = wave.to(device), scal.to(device)
        p_sum = None
        mp_sum = sp_sum = dp_sum = None
        for m in models:
            logit, mp, sp, dp = m(wave, scal)
            p = torch.sigmoid(logit / temperature)
            p_sum = p if p_sum is None else p_sum + p
            mp_sum = mp if mp_sum is None else mp_sum + mp
            sp_sum = sp if sp_sum is None else sp_sum + sp
            dp_sum = dp if dp_sum is None else dp_sum + dp
        probs_l.extend((p_sum / k).cpu().numpy())
        labels_l.extend(labels.numpy())
        mp_l.append((mp_sum / k).cpu()); mt_l.append(map_t)
        sp_l.append((sp_sum / k).cpu()); st_l.append(sbp_t)
        dp_l.append((dp_sum / k).cpu()); dt_l.append(dbp_t)

    y_labels = np.array(labels_l)
    y_probs = np.array(probs_l)
    map_p, map_t = torch.cat(mp_l), torch.cat(mt_l)
    sbp_p, sbp_t = torch.cat(sp_l), torch.cat(st_l)
    dbp_p, dbp_t = torch.cat(dp_l), torch.cat(dt_l)

    cls_m = compute_cls_metrics(y_labels, y_probs, threshold=threshold)
    metrics = dict(
        **cls_m,
        mae_map=(map_p - map_t).abs().mean().item() * 100,
        mae_sbp=(sbp_p - sbp_t).abs().mean().item() * 100,
        mae_dbp=(dbp_p - dbp_t).abs().mean().item() * 100,
        r_map=pearson_r(map_p, map_t),
        r_sbp=pearson_r(sbp_p, sbp_t),
        r_dbp=pearson_r(dbp_p, dbp_t))
    bp_data = dict(
        map_p=(map_p * 100).numpy(), map_t=(map_t * 100).numpy(),
        sbp_p=(sbp_p * 100).numpy(), sbp_t=(sbp_t * 100).numpy(),
        dbp_p=(dbp_p * 100).numpy(), dbp_t=(dbp_t * 100).numpy())
    return metrics, y_labels, y_probs, bp_data

## Cell 8 — Single-fold training loop

Trains one fold with two parameter groups: encoder at a low LR (`1e-5`, gentle fine-tune) and heads at a normal LR (`1e-4`). Scheduler: `CosineAnnealingWarmRestarts(T_0=10, T_mult=2)`. Gradient clipping at 1.0.

Checkpoint selection is by a **combined metric**: `0.5 × AUPRC + 0.5 × scaled mean BP-r` — uses AUPRC instead of AUROC because the positive class is rare. Early stopping after `cfg['patience']` epochs without improvement.

**No threshold or temperature is fit here** — those happen globally later, on pooled OOF predictions across all folds.

In [ ]:
# ==============================================================================
# CELL 15: SINGLE-FOLD TRAIN LOOP
#
# No threshold/calibration here anymore — those are done ONCE, globally, on the
# pooled out-of-fold predictions in run_horizon_sweep. Checkpoint selection
# now uses AUPRC (+ scaled regression r) instead of AUROC.
# ==============================================================================
def train_one_fold(model, train_loader, val_loader, criterion, cfg, save_path):
    model = model.to(DEVICE)

    encoder_params = list(model.encoder.parameters())
    encoder_ids = {id(p) for p in encoder_params}
    head_params = [p for p in model.parameters() if id(p) not in encoder_ids]

    optimizer = torch.optim.AdamW([
        {'params': encoder_params, 'lr': cfg['lr_encoder']},
        {'params': head_params, 'lr': cfg['lr_heads']},
    ], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    w_auprc = cfg.get('select_w_auprc', 0.5)
    w_reg = cfg.get('select_w_reg', 0.5)

    def combined_metric(auprc, r_map, r_sbp, r_dbp):
        r_scaled = ((r_map + r_sbp + r_dbp) / 3.0 + 1.0) / 2.0   # [-1,1]->[0,1]
        return w_auprc * auprc + w_reg * r_scaled

    best_combined, no_improve, best_metrics = -1.0, 0, {}

    for epoch in range(cfg['epochs']):
        model.train()
        for (wave, scal), (labels, map_t, sbp_t, dbp_t) in train_loader:
            wave, scal = wave.to(DEVICE), scal.to(DEVICE)
            labels = labels.to(DEVICE)
            map_t, sbp_t, dbp_t = map_t.to(DEVICE), sbp_t.to(DEVICE), dbp_t.to(DEVICE)

            optimizer.zero_grad()
            cls_logit, mp, sp, dp = model(wave, scal)
            loss, *_ = criterion(cls_logit, mp, sp, dp, labels, map_t, sbp_t, dbp_t)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        y_l, y_logit, mp, mt, sp, st, dp, dt = _val_forward(model, val_loader, DEVICE)
        y_p = 1.0 / (1.0 + np.exp(-y_logit))                    # uncalibrated probs

        if not np.all(np.isfinite(y_p)):
            print(f"    ep {epoch + 1}: non-finite preds — reloading checkpoint.")
            if os.path.exists(save_path):
                model.load_state_dict(torch.load(save_path, map_location=DEVICE))
            no_improve += 1
            if no_improve >= cfg['patience']:
                break
            continue

        two = len(np.unique(y_l)) > 1
        val_auroc = roc_auc_score(y_l, y_p) if two else 0.5
        val_auprc = average_precision_score(y_l, y_p) if two else 0.5
        val_brier = brier_score_loss(y_l, y_p)
        mae_map = (mp - mt).abs().mean().item() * 100
        mae_sbp = (sp - st).abs().mean().item() * 100
        mae_dbp = (dp - dt).abs().mean().item() * 100
        r_map, r_sbp, r_dbp = pearson_r(mp, mt), pearson_r(sp, st), pearson_r(dp, dt)
        val_combined = combined_metric(val_auprc, r_map, r_sbp, r_dbp)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"    ep {epoch + 1:>3}/{cfg['epochs']}  comb={val_combined:.4f}"
                  f"  AUPRC={val_auprc:.4f}  AUROC={val_auroc:.4f}"
                  f"  Brier={val_brier:.4f}  |  MAE {mae_map:.1f}/{mae_sbp:.1f}/"
                  f"{mae_dbp:.1f}  r {r_map:.2f}/{r_sbp:.2f}/{r_dbp:.2f}")

        if val_combined > best_combined:
            best_combined, no_improve = val_combined, 0
            best_metrics = dict(
                auroc=val_auroc, auprc=val_auprc, brier=val_brier,
                combined=val_combined, mae_map=mae_map, mae_sbp=mae_sbp,
                mae_dbp=mae_dbp, r_map=r_map, r_sbp=r_sbp, r_dbp=r_dbp,
                epoch=epoch + 1)
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= cfg['patience']:
                print(f"    Early stop at epoch {epoch + 1} "
                      f"(no improvement for {cfg['patience']} epochs).")
                break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))   # best ckpt
    print(f"    [saved ep {best_metrics.get('epoch', '?')}]  "
          f"AUPRC={best_metrics.get('auprc', float('nan')):.4f}  "
          f"AUROC={best_metrics.get('auroc', float('nan')):.4f}")
    return best_metrics


## Cell 9 — Horizon sweep

Outer loop over forecasting horizons (`offset_3m … offset_15m`). For each horizon: build 5 stratified CV folds, train one model per fold, and write its OOF logits into a pooled-OOF array.

Once all folds finish, fit one temperature on the pooled OOF logits, pick one threshold by Youden's J on the pooled OOF probabilities, then ensemble all five models on the test set with that temperature + threshold. Reports CV mean ± std, pooled OOF AUPRC, and test-set metrics with bootstrap CIs.

In [ ]:
# ==============================================================================
# CELL 16: HORIZON SWEEP  (pooled OOF calibration + threshold + fold ensemble)
# ==============================================================================
def run_horizon_sweep(cfg):
    os.makedirs('/kaggle/working/hybrid_ckpts', exist_ok=True)
    all_results, all_bp_data = [], {}

    for window in TARGET_WINDOWS:
        print(f"\n{'=' * 70}\n  RUN: ppg+abp+ecg   window={window}\n{'=' * 70}")
        t0 = time.time()

        pooled_ds, pooled_labels, folds, test_ds = build_cv_pools_mm(
            cfg['physio_memmap_dir'], window, n_folds=cfg['n_folds'])
        test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'],
                                 shuffle=False, num_workers=2, pin_memory=True)

        fold_metrics = defaultdict(list)
        fold_ckpts = []
        n_pooled = len(pooled_labels)
        oof_logits = np.full(n_pooled, np.nan, dtype=np.float64)   # CHANGE 1/2
        oof_labels = np.array(pooled_labels, dtype=int)

        for fold_idx, (train_idx, val_idx) in enumerate(folds):
            print(f"\n  -- Fold {fold_idx + 1}/{cfg['n_folds']} "
                  f"(train={len(train_idx)}, val={len(val_idx)}) --")

            fold_train_labels = [pooled_labels[i] for i in train_idx]
            train_loader = make_train_loader(
                pooled_ds, train_idx, fold_train_labels, cfg)
            val_loader = DataLoader(Subset(pooled_ds, val_idx),
                                    batch_size=cfg['batch_size'], shuffle=False,
                                    num_workers=2, pin_memory=True)

            criterion_cls = build_cls_criterion(cfg, fold_train_labels)
            criterion = MultiTaskLoss(criterion_cls, alpha=cfg['alpha'],
                                      beta=cfg['beta'],
                                      huber_delta=cfg['huber_delta'])

            model = build_model(cfg)
            ckpt_path = f"/kaggle/working/hybrid_ckpts/{window}_fold{fold_idx + 1}.pth"
            fm = train_one_fold(model, train_loader, val_loader, criterion,
                                cfg, save_path=ckpt_path)
            fold_ckpts.append(ckpt_path)

            for k, v in fm.items():
                if k == 'epoch':
                    continue
                fold_metrics[k].append(v)

            # OOF logits: each pooled sample scored by the fold where it was val
            y_l, y_logit, *_ = _val_forward(model, val_loader, DEVICE)
            oof_logits[val_idx] = y_logit
            del model
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()

        # ── CV summary ─────────────────────────────────────────────────────
        cv_summary = {f'cv_{k}_mean': float(np.mean(v))
                      for k, v in fold_metrics.items()}
        cv_summary.update({f'cv_{k}_std': float(np.std(v))
                           for k, v in fold_metrics.items()})
        print(f"\n  CV Summary [{window}]:")
        for m in ['auprc', 'auroc', 'brier', 'mae_map', 'mae_sbp', 'mae_dbp',
                  'r_map', 'r_sbp', 'r_dbp']:
            if f'cv_{m}_mean' in cv_summary:
                print(f"    {m:>9s}: {cv_summary[f'cv_{m}_mean']:.4f} "
                      f"+/- {cv_summary[f'cv_{m}_std']:.4f}")

        # ── CHANGE 1: calibrate + pick threshold on pooled OOF ─────────────
        valid = ~np.isnan(oof_logits)
        temperature = (fit_temperature(oof_logits[valid], oof_labels[valid])
                       if cfg['calibrate'] else 1.0)
        oof_cal_probs = 1.0 / (1.0 + np.exp(-oof_logits[valid] / temperature))
        chosen_thr, thr_info = pick_threshold_youden(oof_labels[valid], oof_cal_probs)
        oof_auprc = average_precision_score(oof_labels[valid], oof_cal_probs)
        print(f"  [calib] T={temperature:.3f}  |  [thr] {thr_info['method']} "
              f"thr={chosen_thr:.4f}  |  pooled OOF AUPRC={oof_auprc:.4f}")

        # ── CHANGE 2: ensemble all folds on the test set ───────────────────
        models = []
        for ckpt in fold_ckpts:
            m = build_model(cfg)
            m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
            models.append(m)

        test_m, test_labels, test_probs, bp_data = evaluate_ensemble(
            models, test_loader, DEVICE, temperature=temperature,
            threshold=chosen_thr)
        boot_ci = bootstrap_ci(test_labels, test_probs, threshold=chosen_thr,
                               n_bootstrap=1000, ci=0.95, seed=SEED)
        for m in models:
            del m
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

        all_bp_data[window] = dict(bp_data=bp_data, y_labels=test_labels,
                                   y_probs=test_probs, threshold=chosen_thr,
                                   temperature=temperature)

        elapsed = time.time() - t0
        print(f"\n  Test [{window}]  (ensemble of {len(fold_ckpts)}, "
              f"T={temperature:.3f}, thr={chosen_thr:.4f}):")
        print(f"    AUROC = {test_m['auroc']:.4f}  "
              f"[{boot_ci['auroc_lo']:.4f}-{boot_ci['auroc_hi']:.4f}]")
        print(f"    AUPRC = {test_m['auprc']:.4f}  "
              f"[{boot_ci['auprc_lo']:.4f}-{boot_ci['auprc_hi']:.4f}]")
        print(f"    F1    = {test_m['f1']:.4f}  "
              f"[{boot_ci['f1_lo']:.4f}-{boot_ci['f1_hi']:.4f}]")
        print(f"    Brier = {test_m['brier']:.4f}  "
              f"[{boot_ci['brier_lo']:.4f}-{boot_ci['brier_hi']:.4f}]")
        print(f"    MAE  MAP/SBP/DBP = {test_m['mae_map']:.2f}/"
              f"{test_m['mae_sbp']:.2f}/{test_m['mae_dbp']:.2f} mmHg")
        print(f"    r    MAP/SBP/DBP = {test_m['r_map']:.3f}/"
              f"{test_m['r_sbp']:.3f}/{test_m['r_dbp']:.3f}")
        print(f"  Elapsed: {elapsed / 60:.1f} min")

        all_results.append(dict(
            combo_name='ppg+abp+ecg', window=window, n_channels=3,
            elapsed_s=elapsed, temperature=temperature,
            oof_auprc=oof_auprc, **cv_summary,
            **{f'test_{k}': v for k, v in test_m.items()},
            **{f'test_{k}': v for k, v in boot_ci.items()}))

    return all_results, all_bp_data

## Cell 10 — `main()` and figure generation

Runs the sweep, writes results to `hybrid_results.csv`, and produces two plot directories:

- `hybrid_figures/metrics_vs_horizon.png` — AUROC + AUPRC and BP-r vs horizon
- `hybrid_confusion/{window}_cm.png` — confusion matrix per horizon at the calibrated clinical threshold

In [ ]:
# ==============================================================================
# CELL 17/18: EXECUTE + COMPACT PLOTS
# ==============================================================================
def main():
    # compile_physio_to_memmap(...)  # run once if memmaps not built yet
    results, bp_data = run_horizon_sweep(CONFIG)

    df_res = pd.DataFrame(results)
    df_res.to_csv('/kaggle/working/hybrid_results.csv', index=False)
    print(f"\n[OK] Results -> /kaggle/working/hybrid_results.csv "
          f"({len(df_res)} rows)")

    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
        sns.set_theme(style='whitegrid', font_scale=1.05)

        win2min = {'offset_3m': 3, 'offset_5m': 5, 'offset_7m': 7,
                   'offset_10m': 10,'offset_12m': 12, 'offset_15m': 15}
        df_res['horizon_min'] = df_res['window'].map(win2min)
        df_res = df_res.sort_values('horizon_min').reset_index(drop=True)
        os.makedirs('/kaggle/working/hybrid_figures', exist_ok=True)
        os.makedirs('/kaggle/working/hybrid_confusion', exist_ok=True)

        # AUROC + AUPRC + r vs horizon
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        axes[0].plot(df_res['horizon_min'], df_res['test_auroc'],
                     'o-', label='AUROC', color='#5B7FBE', lw=2)
        axes[0].plot(df_res['horizon_min'], df_res['test_auprc'],
                     's--', label='AUPRC', color='#CC5500', lw=2)
        axes[0].set_xlabel('Horizon (min)'); axes[0].set_ylabel('Score')
        axes[0].set_title('Classification vs Horizon', fontweight='bold')
        axes[0].legend(); axes[0].grid(True, ls='--', alpha=0.6)
        for col, name, c in [('test_r_map', 'MAP', '#1f77b4'),
                             ('test_r_sbp', 'SBP', '#d62728'),
                             ('test_r_dbp', 'DBP', '#2ca02c')]:
            axes[1].plot(df_res['horizon_min'], df_res[col], 'o-',
                         label=name, color=c, lw=2)
        axes[1].set_xlabel('Horizon (min)'); axes[1].set_ylabel('Pearson r')
        axes[1].set_title('BP r vs Horizon', fontweight='bold')
        axes[1].legend(); axes[1].grid(True, ls='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig('/kaggle/working/hybrid_figures/metrics_vs_horizon.png',
                    dpi=160, bbox_inches='tight', facecolor='white')
        plt.close()

        # Confusion matrices at the calibrated clinical threshold
        for window, d in bp_data.items():
            thr = d['threshold']
            cm = confusion_matrix(d['y_labels'].astype(int),
                                  (d['y_probs'] >= thr).astype(int))
            fig, ax = plt.subplots(figsize=(5, 4))
            ConfusionMatrixDisplay(
                cm, display_labels=['No Hypo', 'Hypo']).plot(
                ax=ax, colorbar=False, cmap='Blues')
            ax.set_title(f'{window} | thr={thr:.3f} | T={d["temperature"]:.2f}',
                         fontweight='bold', fontsize=10)
            plt.tight_layout()
            plt.savefig(f'/kaggle/working/hybrid_confusion/{window}_cm.png',
                        dpi=150, bbox_inches='tight')
            plt.close()
        print('[OK] figures -> /kaggle/working/hybrid_figures + hybrid_confusion')
    except Exception as e:
        print(f"[warn] plotting skipped: {e}")

    return df_res, bp_data




## Cell 11 — Entry point

Standard `__main__` guard. Calling `main()` directly is also fine in a notebook.

In [ ]:
if __name__ == '__main__':
    main()